# Three Models to Predict

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression, PoissonRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report

# ==========================================
# 0. Data Preparation & Encoding
# ==========================================

#### Make a copy of df and select relevant features for modeling

In [5]:
# Load the attached updated dataset file into df
df = pd.read_csv('updated_dataset.csv')

# Make a copy of df and select relevant features for modeling
model_df = df.copy()

# Select features to include (dropping identifiers like Customer ID)
features = ['Age', 'Gender', 'Category', 'Purchase Amount (USD)', 'Review Rating', 
            'Previous Purchases', 'Subscription Status', 'Discount Applied', 'Shipping Type']

model_df = model_df[[f for f in features if f in model_df.columns]] #retains the validated columns from your list, dropping all other unneeded columns from memory

model_df

,Age,Gender,Category,Purchase Amount (USD),Review Rating,Previous Purchases,Subscription Status,Discount Applied,Shipping Type
0,55,Male,Clothing,53,3.1,14,Yes,Yes,Express
1,19,Male,Clothing,64,3.1,2,Yes,Yes,Express
2,50,Male,Clothing,73,3.1,23,Yes,Yes,Free Shipping
3,21,Male,Footwear,90,3.5,49,Yes,Yes,Next Day Air
4,45,Male,Clothing,49,2.7,31,Yes,Yes,Free Shipping
...,...,...,...,...,...,...,...,...,...
3895,40,Female,Clothing,28,4.2,32,No,No,2-Day Shipping
3896,52,Female,Accessories,49,4.5,41,No,No,Store Pickup
3897,46,Female,Accessories,33,2.9,24,No,No,Standard
3898,44,Female,Footwear,77,3.8,24,No,No,Express


In [6]:
# Convert categorical columns to numeric using One-Hot Encoding
model_df_encoded = pd.get_dummies(model_df, drop_first=True)  # breaks columns into more parts

model_df_encoded

,Age,Purchase Amount (USD),Review Rating,Previous Purchases,Gender_Male,Category_Clothing,Category_Footwear,Category_Outerwear,Subscription Status_Yes,Discount Applied_Yes,Shipping Type_Express,Shipping Type_Free Shipping,Shipping Type_Next Day Air,Shipping Type_Standard,Shipping Type_Store Pickup
0,55,53,3.1,14,True,True,False,False,True,True,True,False,False,False,False
1,19,64,3.1,2,True,True,False,False,True,True,True,False,False,False,False
2,50,73,3.1,23,True,True,False,False,True,True,False,True,False,False,False
3,21,90,3.5,49,True,False,True,False,True,True,False,False,True,False,False
4,45,49,2.7,31,True,True,False,False,True,True,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,40,28,4.2,32,False,True,False,False,False,False,False,False,False,False,False
3896,52,49,4.5,41,False,False,False,False,False,False,False,False,False,False,True
3897,46,33,2.9,24,False,False,False,False,False,False,False,False,False,True,False
3898,44,77,3.8,24,False,False,True,False,False,False,True,False,False,False,False


In [7]:
print(f"Prepared dataset shape for modeling: {model_df_encoded.shape}")

Prepared dataset shape for modeling: (3900, 15)


# ==========================================
# MODEL 1: Predicting Continuous Target ('Purchase Amount (USD)')
# ==========================================

In [8]:

print("\n--- Model 1: Random Forest Regressor (Purchase Amount) ---")

X1 = model_df_encoded.drop(columns=['Purchase Amount (USD)'])
y1 = model_df_encoded['Purchase Amount (USD)']

X_train1, X_test1, y_train1, y_test1 = train_test_split(X1, y1, test_size=0.2, random_state=42)

rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train1, y_train1)

y_pred1 = rf_regressor.predict(X_test1)
print(f"RMSE: {np.sqrt(mean_squared_error(y_test1, y_pred1)):.2f}")
print(f"R2 Score: {r2_score(y_test1, y_pred1):.4f}")


--- Model 1: Random Forest Regressor (Purchase Amount) ---
RMSE: 24.24
R2 Score: -0.0503


Here is a clear explanation of what these metrics mean for Model 1 (Random Forest Regressor predicting Purchase Amount):

1. RMSE: 24.24 (Root Mean Squared Error)
What it means: RMSE measures the average distance between the model's predicted purchase amounts and the actual purchase amounts, in the same units as your target variable (USD).

How to interpret it: On average, your model's predictions for the purchase amount are off by about $24.24.

Context: You need to compare this error against the range or standard deviation of your target variable (Purchase Amount (USD)). If purchases in your dataset typically range from $20 to $100, an error of $24 is quite high.

2. R² Score: -0.0503 (Coefficient of Determination)
What it means: The R² score tells you how well your features explain the variance in the target variable compared to a simple horizontal baseline (predicting the exact mean purchase amount for every customer).

How to interpret it:

A perfect model has an R² of 1.0.

A baseline model that just guesses the average purchase amount has an R² of 0.0.

A negative R² (like -0.0503) means your model is performing worse than simply guessing the average purchase amount.

Why is the R² negative? (The Takeaway)
A negative R² strongly suggests that the features currently selected (Age, Gender, Category, Review Rating, etc.) have little to no linear or pattern-based relationship with the Purchase Amount (USD) in this specific dataset. In retail datasets like this, purchase amounts are often distributed relatively uniformly or randomly across different categories and ages, making them very difficult for a machine learning model to predict accurately from demographics alone.


# ==========================================
# MODEL 2: Count Regression ('Previous Purchases')
# ==========================================

In [9]:

print("\n--- Model 2: Poisson Regressor (Previous Purchases) ---")

X2 = model_df_encoded.drop(columns=['Previous Purchases'])
y2 = model_df_encoded['Previous Purchases']

X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2, random_state=42)

poisson_reg = PoissonRegressor(max_iter=500)
poisson_reg.fit(X_train2, y_train2)

y_pred2 = poisson_reg.predict(X_test2)
print(f"Poisson RMSE: {np.sqrt(mean_squared_error(y_test2, y_pred2)):.2f}")
print(f"Poisson R2 Score: {r2_score(y_test2, y_pred2):.4f}")


--- Model 2: Poisson Regressor (Previous Purchases) ---
Poisson RMSE: 14.14
Poisson R2 Score: 0.0011


Here is a clear explanation of what these metrics mean for Model 2 (Poisson Regressor predicting Previous Purchases):

1. Poisson RMSE: 14.14 (Root Mean Squared Error)
What it means: This represents the average deviation between the model's predicted number of previous purchases and the actual counts, measured in the same units (number of purchases).

How to interpret it: On average, the model's count predictions are off by about 14.14 purchases.

2. Poisson R² Score: 0.0011 (Coefficient of Determination)
What it means: An R² score close to 0 (in this case, 0.0011) indicates that the model explains almost none of the variance in the target variable (Previous Purchases).

How to interpret it:

A score of 0.0 means the model performs identically to just guessing the average number of previous purchases for every customer.

A score of 0.0011 is barely above zero, meaning the features provided (Age, Gender, Category, etc.) contain virtually no predictive signal for how many past purchases a customer has made.

Why is the R² so close to zero?
In many consumer retail datasets, a customer's history of past purchases (Previous Purchases) behaves like an independent random count or uniform distribution relative to demographics like age or item category. Because the input features don't share a strong mathematical correlation with the frequency of past shopping behavior, the Poisson model struggles to find a meaningful pattern to learn from.

# ==========================================
# MODEL 3: Logistic Regression (Binary Target: Subscription Status)
# ==========================================

In [10]:

print("\n--- Model 3: Logistic Regression (Subscription Status) ---")

sub_col = [col for col in model_df_encoded.columns if 'Subscription Status' in col][0]

X3 = model_df_encoded.drop(columns=[sub_col])
y3 = model_df_encoded[sub_col]

X_train3, X_test3, y_train3, y_test3 = train_test_split(X3, y3, test_size=0.2, random_state=42)

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train3, y_train3)

y_pred3 = log_reg.predict(X_test3)
print(f"Accuracy: {accuracy_score(y_test3, y_pred3):.4f}")
print("\nClassification Report:")
print(classification_report(y_test3, y_pred3))


--- Model 3: Logistic Regression (Subscription Status) ---
Accuracy: 0.8256

Classification Report:
              precision    recall  f1-score   support

       False       1.00      0.76      0.86       558
        True       0.62      1.00      0.77       222

    accuracy                           0.83       780
   macro avg       0.81      0.88      0.81       780
weighted avg       0.89      0.83      0.83       780



Here is a clear breakdown of what your Logistic Regression classification metrics mean for predicting Subscription Status:

1. Overall Accuracy: 82.56% (or 0.83)
What it means: Out of all the test customers evaluated, the model correctly predicted whether they had a subscription about 83% of the time.

Context: While an 83% accuracy sounds quite high, it's important to look closer at the individual classes because datasets often have an imbalance between subscribers and non-subscribers.

2. Breakdown by Class (False vs. True)
The classification report breaks performance down into two groups: False (non-subscribers) and True (subscribers).

Precision (When the model makes a guess, how often is it right?):

False (1.00): When the model predicts a customer is not a subscriber, it is 100% correct.

True (0.62): When the model predicts a customer is a subscriber, it is correct 62% of the time (meaning about 38% are "false alarms" or false positives).

Recall / Sensitivity (Out of all actual instances, how many did the model find?):

False (0.76): The model successfully caught 76% of all actual non-subscribers, missing 24%.

True (1.00): The model successfully caught 100% of all actual subscribers (it missed none!).

F1-Score (The harmonic balance between Precision and Recall):

False (0.86) and True (0.77) give a combined performance metric for each class, showing the model is exceptionally good at identifying non-subscribers and reasonably strong at finding subscribers.

The Big Picture Takeaway
Unlike the regression models we looked at earlier (which struggled to find patterns), this Logistic Regression model is actually doing a solid job. It achieves high overall accuracy and manages to successfully identify every single actual subscriber (Recall of 1.00 for the True class), though it occasionally mislabels a non-subscriber as a subscriber (Precision of 0.62).

# Conclusion:

Evaluating whether these models are helpful requires looking at them through a business and practical lens. 

In data science, a model's "helpfulness" depends entirely on its performance and how its predictions
translate into business value.Here is how each model performs and how they can be applied in a 
real-world retail or e-commerce business case:1. 

**Model 3: Logistic Regression (Subscription Status) — Highly Helpful**
    
**Business Value: High.** Because this model achieved solid accuracy (~83%) and successfully 
caught 100% of actual subscribers, it gives the business a reliable tool to identify customer patterns. 
 
**Real-World Application:**

**Targeted Marketing Campaigns:** Instead of wasting money marketing subscriptions 
to every customer, the company can run the model on active users to find non-subscribers who exhibit 
"subscriber-like" traits.

**Retention and Upselling:** You can proactively offer exclusive perks, 
free shipping, or targeted discounts to high-probability non-subscribers to nudge them into signing up 
for a subscription program, directly increasing recurring revenue.

**2. Models 1 & 2: Purchase Amount & Previous Purchases — Not Helpful in Current State (A Real-World 
Lesson)**

**Business Value:** Low to None (due to near-zero or negative R² scores).

Real-World Application / The Reality Check:

In a real business setting, **you cannot rely on these models for forecasting.** If a model’s R² score is 
close to 0 or negative, it means knowing a customer's age, gender, or product category tells you 
almost nothing about how much they will spend or how many times they have shopped before.

**How to make them helpful in the real world:** To make these models practically useful, you would need to 
feed them behavioral features rather than just basic demographics—such as:

Recency, Frequency, Monetary (RFM) metrics.
Website browsing history or time spent on site.
Past discount usage history or seasonal buying patterns.

**Summary Takeaway**
In a real enterprise environment, Model 3 is ready to be 
deployed for customer segmentation and marketing optimization. Meanwhile, **Models 1 and 2** serve as a 
critical checkpoint: they show that basic demographic data alone isn't enough to predict customer 
spending habits, signaling that your data engineering pipeline needs richer behavioral data before 
those specific machine learning predictions can become useful.
